In [ ]:
"""
Preprocess MSD Task06_Lung NIfTI volumes into slice-level .npy files.

Reads the original Medical Segmentation Decathlon Lung dataset (NIfTI CT
volumes with known Hounsfield Units), applies lung-window normalisation,
and saves axial slices as .npy files with manifest JSONs for balanced
training.

To save disk space on Kaggle:
  - Train: only slices in the 50:50 balanced set + their i-1/i+1 neighbours
    (needed for 2.5D triplet context) are saved.
  - Val / Test: all slices are saved.

Output is compatible with both:
  - 2D dataset (single-slice, 1-channel)
  - 2.5D dataset (triplet [i-1, i, i+1], 3-channel)

Usage:
    python preprocess_msd_lung.py
"""

import json
import os
import random
import shutil
from pathlib import Path

import nibabel as nib
import numpy as np
from scipy.ndimage import zoom

# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
SEED = 42
TASK_DIR = os.path.join(os.path.dirname(os.path.abspath(__file__)), "Task06_Lung")
OUTPUT_DIR = os.path.join(os.path.dirname(os.path.abspath(__file__)), "lung_msd_processed")

HU_MIN = -1024.0
HU_MAX = 600.0

TARGET_SIZE = 256  # resize each axial slice to (TARGET_SIZE, TARGET_SIZE)

N_VAL = 6
N_TEST = 7

random.seed(SEED)
np.random.seed(SEED)

# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------

def hu_normalise(arr: np.ndarray) -> np.ndarray:
    """Clip to lung window and rescale to [0, 1]."""
    clipped = np.clip(arr.astype(np.float32), HU_MIN, HU_MAX)
    return ((clipped - HU_MIN) / (HU_MAX - HU_MIN)).astype(np.float32)


def resize_slice(arr: np.ndarray, target: int, order: int = 1) -> np.ndarray:
    """Resize a 2D slice to (target, target). order=1 bilinear, order=0 nearest."""
    h, w = arr.shape
    if h == target and w == target:
        return arr
    scale = (target / h, target / w)
    return zoom(arr, scale, order=order).astype(np.float32)


def load_dataset_json(task_dir: str):
    """Parse dataset.json and return list of (patient_id, img_path, lbl_path)."""
    with open(os.path.join(task_dir, "dataset.json")) as f:
        ds = json.load(f)

    pairs = []
    for entry in ds["training"]:
        img_path = os.path.join(task_dir, entry["image"].lstrip("./"))
        lbl_path = os.path.join(task_dir, entry["label"].lstrip("./"))
        patient_id = Path(img_path).stem.replace(".nii", "")
        pairs.append((patient_id, img_path, lbl_path))
    return pairs


# ---------------------------------------------------------------------------
# Per-split processing
# ---------------------------------------------------------------------------

def process_split(split_name, split_pairs, manifest_dir):
    split_out = os.path.join(OUTPUT_DIR, split_name)
    os.makedirs(split_out, exist_ok=True)

    trim = split_name == "train"

    # ------------------------------------------------------------------
    # Pass 1 – scan labels only (tiny files) to collect metadata
    # ------------------------------------------------------------------
    print(f"\n{'='*60}")
    print(f"  {split_name.upper()} – Pass 1: scanning labels")
    print(f"{'='*60}")

    all_meta = []
    patient_num_slices = {}

    for patient_id, _img_path, lbl_path in sorted(split_pairs, key=lambda x: x[0]):
        lbl_nii = nib.as_closest_canonical(nib.load(lbl_path))
        lbl_data = lbl_nii.get_fdata(dtype=np.float32)
        num_slices = lbl_data.shape[2]
        patient_num_slices[patient_id] = num_slices

        tumor_count = 0
        for s in range(num_slices):
            has_tumor = bool((lbl_data[:, :, s] > 0).any())
            if has_tumor:
                tumor_count += 1
            all_meta.append({
                "patient": patient_id,
                "slice_idx": s,
                "num_slices": num_slices,
                "has_tumor": has_tumor,
            })
        print(f"    {patient_id}: {num_slices} slices, {tumor_count} with tumor")

    # ------------------------------------------------------------------
    # Compute which slices to save
    # ------------------------------------------------------------------
    tumor_indices = [i for i, m in enumerate(all_meta) if m["has_tumor"]]
    non_tumor_indices = [i for i, m in enumerate(all_meta) if not m["has_tumor"]]

    if split_name != "test":
        n_tumor = len(tumor_indices)
        sampled_neg = random.sample(non_tumor_indices, min(n_tumor, len(non_tumor_indices)))
        balanced_raw_indices = tumor_indices + sampled_neg
        balanced_keys = {
            (all_meta[i]["patient"], all_meta[i]["slice_idx"])
            for i in balanced_raw_indices
        }
    else:
        balanced_keys = set()

    if trim:
        # Only save balanced slices + their i-1/i+1 neighbours
        save_keys = set()
        for idx in balanced_raw_indices:
            m = all_meta[idx]
            p, s, n = m["patient"], m["slice_idx"], m["num_slices"]
            for offset in (-1, 0, 1):
                save_keys.add((p, max(0, min(n - 1, s + offset))))
    else:
        save_keys = {(m["patient"], m["slice_idx"]) for m in all_meta}

    # Group by patient for pass 2
    save_by_patient = {}
    for p, s in save_keys:
        save_by_patient.setdefault(p, set()).add(s)

    # ------------------------------------------------------------------
    # Pass 2 – load full volumes and save selected slices
    # ------------------------------------------------------------------
    print(f"\n  {split_name.upper()} – Pass 2: saving slices")

    for patient_id, img_path, lbl_path in sorted(split_pairs, key=lambda x: x[0]):
        if patient_id not in save_by_patient:
            continue

        img_nii = nib.as_closest_canonical(nib.load(img_path))
        lbl_nii = nib.as_closest_canonical(nib.load(lbl_path))
        img_data = img_nii.get_fdata(dtype=np.float32)
        lbl_data = lbl_nii.get_fdata(dtype=np.float32)

        voxel_spacing = img_nii.header.get_zooms()
        num_slices = img_data.shape[2]
        slices_to_save = sorted(save_by_patient[patient_id])

        print(f"    {patient_id}: shape={img_data.shape}  "
              f"spacing={tuple(round(float(s), 2) for s in voxel_spacing)}  "
              f"HU=[{img_data.min():.0f},{img_data.max():.0f}]  "
              f"saving {len(slices_to_save)}/{num_slices}")

        dst_data = os.path.join(split_out, patient_id, "data")
        dst_masks = os.path.join(split_out, patient_id, "masks")
        os.makedirs(dst_data, exist_ok=True)
        os.makedirs(dst_masks, exist_ok=True)

        for s in slices_to_save:
            ct_norm = hu_normalise(img_data[:, :, s])
            mask_bin = (lbl_data[:, :, s] > 0).astype(np.float32)
            np.save(os.path.join(dst_data, f"{s}.npy"),
                    resize_slice(ct_norm, TARGET_SIZE, order=1))
            np.save(os.path.join(dst_masks, f"{s}.npy"),
                    resize_slice(mask_bin, TARGET_SIZE, order=0))

    # ------------------------------------------------------------------
    # Build manifest with only saved slices
    # ------------------------------------------------------------------
    saved_samples = [m for m in all_meta if (m["patient"], m["slice_idx"]) in save_keys]

    saved_tumor = [i for i, m in enumerate(saved_samples) if m["has_tumor"]]
    saved_non_tumor = [i for i, m in enumerate(saved_samples) if not m["has_tumor"]]

    manifest = {
        "split": split_name,
        "patients": sorted(set(m["patient"] for m in saved_samples)),
        "samples": saved_samples,
        "tumor_indices": saved_tumor,
        "non_tumor_indices": saved_non_tumor,
    }

    if split_name != "test":
        key_to_saved_pos = {}
        for i, m in enumerate(saved_samples):
            key_to_saved_pos[(m["patient"], m["slice_idx"])] = i

        balanced_indices = [
            key_to_saved_pos[k] for k in balanced_keys if k in key_to_saved_pos
        ]
        random.shuffle(balanced_indices)
        manifest["balanced_indices"] = balanced_indices

    manifest_path = os.path.join(manifest_dir, f"{split_name}_manifest.json")
    with open(manifest_path, "w") as f:
        json.dump(manifest, f, indent=2)

    n_bal = len(manifest.get("balanced_indices", []))
    total_on_disk = sum(len(v) for v in save_by_patient.values())
    print(f"\n  {split_name}: {total_on_disk} slices on disk | "
          f"{len(saved_samples)} in manifest | "
          f"{len(saved_tumor)} tumor | {len(saved_non_tumor)} non-tumor | "
          f"balanced={n_bal}")


# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------

def main():
    pairs = load_dataset_json(TASK_DIR)
    print(f"Found {len(pairs)} labelled volumes in {TASK_DIR}")

    random.shuffle(pairs)
    test_pairs = pairs[:N_TEST]
    val_pairs = pairs[N_TEST: N_TEST + N_VAL]
    train_pairs = pairs[N_TEST + N_VAL:]

    if os.path.isdir(OUTPUT_DIR):
        print(f"Removing existing output: {OUTPUT_DIR}")
        shutil.rmtree(OUTPUT_DIR)

    manifest_dir = os.path.join(OUTPUT_DIR, "manifests")
    os.makedirs(manifest_dir, exist_ok=True)

    for split_name, split_pairs in [("train", train_pairs), ("val", val_pairs), ("test", test_pairs)]:
        process_split(split_name, split_pairs, manifest_dir)

    print(f"\nDone. Output in {OUTPUT_DIR}")

    total_npy = 0
    for root, _dirs, files in os.walk(OUTPUT_DIR):
        for fname in files:
            if fname.endswith(".npy"):
                total_npy += os.path.getsize(os.path.join(root, fname))
    print(f"Total .npy size: {total_npy / (1024**3):.2f} GB")


if __name__ == "__main__":
    main()